# Notebook 5: Decision Tree Model

## Step 1 — Import Required Libraries

Import the Python libraries required for data preprocessing, Decision Tree model development, hyperparameter tuning, and performance evaluation. Pandas and NumPy support data handling, while Scikit-learn provides the Decision Tree algorithm, cross-validation tools, and evaluation metrics.

In [14]:
# IT3091 - Telco Customer Churn
# Member 4: Decision Tree Model

import pandas as pd
import numpy as np

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Libraries imported successfully!")

Libraries imported successfully!


## Step 2 — Load Dataset and Prepare the Data

In this step, we load the shared training and testing datasets prepared by our team. The input features (X) and target labels (y) are loaded separately from CSV files.

The dataset is split into training and testing sets using an 80:20 ratio, containing 5,634 training records and 1,409 testing records. Using the shared split ensures that our Decision Tree model is evaluated consistently with the other models in the project. The preprocessing pipeline is applied later, using the training data to fit the transformations.

In [16]:
# Step 2: Load the shared team train/test split

X_train = pd.read_csv("../data/processed/X_train_raw.csv")
X_test = pd.read_csv("../data/processed/X_test_raw.csv")

y_train = pd.read_csv("../data/processed/y_train.csv")["Churn"]
y_test = pd.read_csv("../data/processed/y_test.csv")["Churn"]

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

Training features: (5634, 19)
Testing features: (1409, 19)
Training labels: (5634,)
Testing labels: (1409,)


## Step 3 — Build the Preprocessing Pipeline and Train the Decision Tree

In this step, we prepare the input features for machine learning. Numerical features are scaled using `StandardScaler`, categorical features are converted into numerical values using `OneHotEncoder`, and the `SeniorCitizen` feature is passed through without additional transformation.

We combine these preprocessing steps and the Decision Tree classifier into a single pipeline. The `class_weight="balanced"` parameter helps the model account for the imbalance between churn and non-churn customers. Finally, we train the pipeline using the training dataset.

In [18]:
# Step 3: Preprocessing Pipeline and Baseline Decision Tree

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

# Define feature columns
numeric_features = ["tenure", "MonthlyCharges", "TotalCharges"]
binary_features = ["SeniorCitizen"]

# Identify categorical features
categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

# Validate required columns
required_features = numeric_features + binary_features
missing_columns = [
    col for col in required_features
    if col not in X_train.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Check missing values
print("Missing values in X_train:", X_train.isnull().sum().sum())
print("Missing values in X_test:", X_test.isnull().sum().sum())

# Create preprocessing pipeline
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    (
        "cat",
        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        categorical_features
    ),
    ("binary", "passthrough", binary_features)
])

# Create Decision Tree pipeline
dt_model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        DecisionTreeClassifier(
            random_state=42,
            class_weight="balanced"
        )
    )
])

# Train the baseline model
dt_model.fit(X_train, y_train)

print("Decision Tree baseline model trained successfully!")

Missing values in X_train: 0
Missing values in X_test: 0
Decision Tree baseline model trained successfully!


C:\Users\ASUS\AppData\Local\Temp\ipykernel_13096\2962590028.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


## Step 4 — Evaluate the Baseline Decision Tree Model

In this step, we evaluate the baseline model using accuracy, precision, recall, F1-score, and a confusion matrix. The model achieved **72.75% accuracy** and an **F1-score of 0.48 for churn customers**. The low churn recall of **0.47** indicates that the model misses many actual churners, motivating further hyperparameter tuning.

In [6]:
# Evaluate Decision Tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = dt_model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7275

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.82      0.82      1035
           1       0.49      0.47      0.48       374

    accuracy                           0.73      1409
   macro avg       0.65      0.65      0.65      1409
weighted avg       0.72      0.73      0.73      1409


Confusion Matrix:
[[849 186]
 [198 176]]


## Step 5 — Hyperparameter Tuning Using GridSearchCV

In this step, we use `GridSearchCV` with 5-fold stratified cross-validation to find the best Decision Tree parameters. We tune `max_depth` and `min_samples_leaf` using the F1-score to improve churn detection and reduce overfitting.

**Best parameters:** `max_depth = 7`, `min_samples_leaf = 50`

**Best cross-validation F1-score:** 0.6251

In [7]:

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        random_state=42,
        class_weight="balanced"
    ))
])

params = {
    "classifier__max_depth": [3, 5, 7, 10, None],
    "classifier__min_samples_leaf": [5, 10, 20, 50]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = GridSearchCV(
    pipe,
    params,
    scoring="f1",
    cv=cv,
    n_jobs=-1
)

search.fit(X_train, y_train)

print("Best parameters:", search.best_params_)
print("Best CV F1:", round(search.best_score_, 4))


Best parameters: {'classifier__max_depth': 7, 'classifier__min_samples_leaf': 50}
Best CV F1: 0.6251


## Step 6 — Evaluate the Tuned Decision Tree Model

In this step, we evaluate the tuned model on the test dataset using the classification report and confusion matrix. The model achieves approximately **75% accuracy** and an **F1-score of 0.62 for churn customers**. Churn recall improves to **0.77**, meaning the model identifies 77% of actual churners in the test set.

The tuned model detects more churners than the baseline model, although some loyal customers are incorrectly classified as churners.

In [8]:
# Evaluate the tuned Decision Tree on test data

best_model = search.best_estimator_

y_pred_tuned = best_model.predict(X_test)

print("Best Parameters:", search.best_params_)
print("\nClassification Report:")
print(classification_report(
    y_test, y_pred_tuned, zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))

Best Parameters: {'classifier__max_depth': 7, 'classifier__min_samples_leaf': 50}

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.74      0.81      1035
           1       0.52      0.77      0.62       374

    accuracy                           0.75      1409
   macro avg       0.71      0.76      0.72      1409
weighted avg       0.80      0.75      0.76      1409


Confusion Matrix:
[[767 268]
 [ 85 289]]


## Step 7 — Export Decision Tree Rules

In this step, we extract readable decision rules from the tuned Decision Tree model using `export_text()`. These rules show how the model uses customer features to predict churn. We save the rules in `reports/decision_tree_rules.txt` for documentation and further analysis.

In [9]:

from sklearn.tree import export_text
from pathlib import Path

tree = best_model.named_steps["classifier"]

feature_names = best_model.named_steps[
    "preprocessor"
].get_feature_names_out()

rules = export_text(
    tree,
    feature_names=list(feature_names)
)

output_path = Path("../reports/decision_tree_rules.txt")
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(rules, encoding="utf-8")

print("Tree rules saved to:", output_path)
print("\nFirst 20 lines:\n")
print("\n".join(rules.splitlines()[:20]))


Tree rules saved to: ..\reports\decision_tree_rules.txt

First 20 lines:

|--- cat__Contract_Month-to-month <= 0.50
|   |--- num__MonthlyCharges <= 0.95
|   |   |--- cat__OnlineSecurity_No <= 0.50
|   |   |   |--- cat__Contract_Two year <= 0.50
|   |   |   |   |--- num__MonthlyCharges <= 0.24
|   |   |   |   |   |--- cat__PaymentMethod_Electronic check <= 0.50
|   |   |   |   |   |   |--- cat__PaymentMethod_Bank transfer (automatic) <= 0.50
|   |   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |   |--- cat__PaymentMethod_Bank transfer (automatic) >  0.50
|   |   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |--- cat__PaymentMethod_Electronic check >  0.50
|   |   |   |   |   |   |--- class: 0
|   |   |   |   |--- num__MonthlyCharges >  0.24
|   |   |   |   |   |--- cat__PaperlessBilling_No <= 0.50
|   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |--- cat__PaperlessBilling_No >  0.50
|   |   |   |   |   |   |--- class: 0
|   |   |   |--- cat__Contract_Two year >

## Step 8 — Analyze Feature Importance

In this step, we calculate and rank the features based on their importance in the trained Decision Tree model. The results are displayed and saved as a CSV file for further analysis.

The most important features are **Month-to-month contract (0.5768)**, **tenure (0.1128)**, and **Fiber optic internet service (0.1024)**. These features have the highest importance scores in this Decision Tree model.

In [10]:

import pandas as pd

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": tree.feature_importances_
}).sort_values("Importance", ascending=False)

print(feature_importance.head(10))

feature_importance.to_csv(
    "../reports/decision_tree_feature_importance.csv",
    index=False
)
print("\nFeature importance saved!")


                                Feature  Importance
35         cat__Contract_Month-to-month    0.576786
0                           num__tenure    0.112818
15     cat__InternetService_Fiber optic    0.102377
1                   num__MonthlyCharges    0.066049
37               cat__Contract_Two year    0.021569
2                     num__TotalCharges    0.020495
26                  cat__TechSupport_No    0.019377
42  cat__PaymentMethod_Electronic check    0.018087
32              cat__StreamingMovies_No    0.015260
17               cat__OnlineSecurity_No    0.012858

Feature importance saved!


## Step 9 — Customer Churn Risk Profiling

In this step, we calculate each test customer's predicted churn probability and classify customers into Low, Medium, and High risk groups. We compare the risk groups with actual churn outcomes and save the results as a CSV file.

The High-risk group contains **474 customers**, with an actual churn rate of **56.1%**. The Medium-risk group has a churn rate of **22.0%**, while the Low-risk group has a churn rate of **6.3%**. These preliminary risk profiles can help identify customer groups for further retention analysis.

In [11]:

# Predict churn risk for test customers
risk_probability = best_model.predict_proba(X_test)[:, 1]

risk_profile = X_test.copy()
risk_profile["Actual_Churn"] = y_test.values
risk_profile["Predicted_Churn_Risk"] = risk_probability

risk_profile["Risk_Level"] = pd.cut(
    risk_profile["Predicted_Churn_Risk"],
    bins=[-0.01, 0.30, 0.60, 1.0],
    labels=["Low", "Medium", "High"]
)

print(risk_profile["Risk_Level"].value_counts())
print("\nRisk level vs actual churn:")
print(pd.crosstab(
    risk_profile["Risk_Level"],
    risk_profile["Actual_Churn"],
    normalize="index"
).round(3))

risk_profile.to_csv(
    "../reports/decision_tree_risk_profile.csv",
    index=False
)

print("\nRisk profile saved!")



Risk_Level
Low       622
High      474
Medium    313
Name: count, dtype: int64

Risk level vs actual churn:
Actual_Churn      0      1
Risk_Level                
Low           0.937  0.063
Medium        0.780  0.220
High          0.439  0.561

Risk profile saved!


# Decision Log

| # | Date | Decision | Options Considered | Reason / Evidence | Owner |
|---|---|---|---|---|---|
| 1 | — | Project track: Guided Data Track | Guided Data Track vs Industry Explorer | Selected based on the project's scope and available time | Group |
| 2 | — | Primary business lens: Churn prediction | Churn prediction vs customer segmentation | The dataset contains a labelled `Churn` target variable | Group |
| 3 | 2026-10-11 | Decision Tree baseline model | Decision Tree vs not using a Decision Tree | Provides an interpretable model for predicting customer churn | M4 |
| 4 | 2026-10-11 | Hyperparameter tuning | Different `max_depth` and `min_samples_leaf` values | GridSearchCV selected `max_depth=7` and `min_samples_leaf=50`, with a best cross-validation F1-score of 0.6251 | M4 |
| 5 | 2026-10-11 | Model evaluation metrics | Accuracy alone vs precision, recall, F1-score and confusion matrix | Recall and F1-score help assess how effectively the model identifies churn customers | M4 |
| 6 | 2026-10-11 | Customer churn risk profiling | Low, Medium and High risk groups | Test-set actual churn rates were 6.3% (Low), 22.0% (Medium) and 56.1% (High) using preliminary risk thresholds | M4 |

## Notes

- The Decision Tree was tuned using five-fold stratified cross-validation.
- The tuned model achieved approximately 75% test accuracy and a churn-class F1-score of 0.62.
- Risk thresholds are preliminary and may require further validation.
- Feature importance and churn risk profiles show associations, not proof of causation.
- Final model profiling, recommendations and reproducibility decisions will be added during project integration.